# Sensor Activity + Fall Detection (CNN + RNN)

Plantilla base para tu proyecto sin datos aun.

- CNN: ventanas de 2 segundos para actividad (`walking`, `running`, `still`, `stairs`).
- Trigger: pico de aceleracion para posible caida.
- RNN: ventana de 20 segundos para confirmar caida por patron temporal.
- Filtros opcionales: pasa-bajas / pasa-altas / pasa-banda.


## 1) Dependencias

Instalar si hace falta:

`pip install numpy pandas scipy scikit-learn matplotlib tensorflow`


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    from scipy.signal import butter, sosfiltfilt
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

print('TensorFlow:', tf.__version__)
print('SciPy disponible:', SCIPY_AVAILABLE)


In [ ]:
@dataclass
class PipelineConfig:
    sample_rate_hz: int = 50
    cnn_window_seconds: int = 2
    rnn_window_seconds: int = 20

    sensor_columns: Tuple[str, ...] = (
        'acc_x', 'acc_y', 'acc_z',
        'gyro_x', 'gyro_y', 'gyro_z'
    )

    activity_classes: Tuple[str, ...] = (
        'walking', 'running', 'still', 'stairs'
    )

    # Trigger inicial para posible caida (m/s^2)
    fall_peak_threshold: float = 22.0

    # Filtro opcional: none | lowpass | highpass | bandpass
    filter_mode: str = 'none'
    filter_order: int = 4
    low_cut_hz: float = 0.3
    high_cut_hz: float = 8.0

    # Training
    batch_size: int = 64
    epochs_cnn: int = 30
    epochs_rnn: int = 30
    learning_rate: float = 1e-3

cfg = PipelineConfig()
CNN_STEPS = cfg.cnn_window_seconds * cfg.sample_rate_hz
RNN_STEPS = cfg.rnn_window_seconds * cfg.sample_rate_hz
N_FEATURES = len(cfg.sensor_columns)

print('CNN steps:', CNN_STEPS)
print('RNN steps:', RNN_STEPS)
print('N features:', N_FEATURES)


In [ ]:
def load_raw_sensor_dataframe() -> pd.DataFrame:
    """
    TODO: conecta tu dataset aqui.

    Recomendado incluir columnas:
    - acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
    - activity_label (CNN)
    - fall_label (RNN binario: 0/1)
    - timestamp, subject_id, session_id (opcional)
    """
    raise NotImplementedError('Define la carga de datos cuando se cierre el esquema')


In [ ]:
def build_sos_filter(cfg: PipelineConfig):
    if cfg.filter_mode == 'none':
        return None
    if not SCIPY_AVAILABLE:
        raise RuntimeError('SciPy no disponible para filtros')

    nyq = 0.5 * cfg.sample_rate_hz
    if cfg.filter_mode == 'lowpass':
        wn = cfg.high_cut_hz / nyq
        return butter(cfg.filter_order, wn, btype='lowpass', output='sos')
    if cfg.filter_mode == 'highpass':
        wn = cfg.low_cut_hz / nyq
        return butter(cfg.filter_order, wn, btype='highpass', output='sos')
    if cfg.filter_mode == 'bandpass':
        wn = [cfg.low_cut_hz / nyq, cfg.high_cut_hz / nyq]
        return butter(cfg.filter_order, wn, btype='bandpass', output='sos')
    raise ValueError(f'filter_mode invalido: {cfg.filter_mode}')


def apply_optional_filter(window: np.ndarray, cfg: PipelineConfig) -> np.ndarray:
    sos = build_sos_filter(cfg)
    if sos is None:
        return window
    return sosfiltfilt(sos, window, axis=0)


def sliding_windows(values: np.ndarray, labels: Optional[np.ndarray], window_size: int, stride: int):
    X = []
    y = [] if labels is not None else None
    n = len(values)

    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        X.append(values[start:end])
        if labels is not None:
            seg = labels[start:end]
            vals, counts = np.unique(seg, return_counts=True)
            y.append(vals[np.argmax(counts)])

    X = np.asarray(X, dtype=np.float32)
    if labels is not None:
        y = np.asarray(y)
    return X, y


def normalize_windows(X_train: np.ndarray, X_val: np.ndarray):
    scaler = StandardScaler()
    n1, t1, f1 = X_train.shape
    n2, t2, f2 = X_val.shape

    X_train_2d = X_train.reshape(-1, f1)
    X_val_2d = X_val.reshape(-1, f2)

    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n1, t1, f1)
    X_val_scaled = scaler.transform(X_val_2d).reshape(n2, t2, f2)
    return X_train_scaled, X_val_scaled, scaler


In [ ]:
def build_cnn_activity_model(input_steps: int, n_features: int, n_classes: int, lr: float = 1e-3):
    inp = keras.Input(shape=(input_steps, n_features), name='sensor_window_2s')
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax', name='activity_class')(x)

    model = keras.Model(inp, out, name='cnn_activity_2s')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


cnn_model = build_cnn_activity_model(CNN_STEPS, N_FEATURES, len(cfg.activity_classes), cfg.learning_rate)
cnn_model.summary()


In [ ]:
def compute_acc_magnitude(window_2s: np.ndarray, feature_names: Tuple[str, ...]) -> np.ndarray:
    ix = feature_names.index('acc_x')
    iy = feature_names.index('acc_y')
    iz = feature_names.index('acc_z')
    acc = window_2s[:, [ix, iy, iz]]
    return np.sqrt(np.sum(acc ** 2, axis=1))


def detect_fall_peak(window_2s: np.ndarray, cfg: PipelineConfig) -> bool:
    mag = compute_acc_magnitude(window_2s, cfg.sensor_columns)
    return float(np.max(mag)) >= cfg.fall_peak_threshold


In [ ]:
def build_rnn_fall_model(input_steps: int, n_features: int, lr: float = 1e-3):
    inp = keras.Input(shape=(input_steps, n_features), name='sensor_window_20s')
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.GRU(64))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid', name='fall_confirmed')(x)

    model = keras.Model(inp, out, name='rnn_fall_20s')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='acc'), keras.metrics.AUC(name='auc')]
    )
    return model


rnn_model = build_rnn_fall_model(RNN_STEPS, N_FEATURES, cfg.learning_rate)
rnn_model.summary()


## 2) Placeholders de entrenamiento (sin datos aun)

Descomenta y conecta cuando definas el dataset.


In [ ]:
# -------- CNN TRAINING SKELETON --------
# df = load_raw_sensor_dataframe()
# values = df[list(cfg.sensor_columns)].values.astype(np.float32)
# y_activity = df['activity_label'].values
#
# X_cnn, y_cnn = sliding_windows(values, y_activity, window_size=CNN_STEPS, stride=CNN_STEPS // 2)
# X_cnn = np.stack([apply_optional_filter(w, cfg) for w in X_cnn], axis=0)
#
# X_train, X_val, y_train, y_val = train_test_split(
#     X_cnn, y_cnn, test_size=0.2, random_state=42, stratify=y_cnn
# )
#
# X_train, X_val, scaler_cnn = normalize_windows(X_train, X_val)
#
# class_ids = np.unique(y_train)
# class_weights = compute_class_weight('balanced', classes=class_ids, y=y_train)
# class_weight_map = {int(k): float(v) for k, v in zip(class_ids, class_weights)}
#
# callbacks = [
#     keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
#     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
# ]
#
# hist_cnn = cnn_model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=cfg.epochs_cnn,
#     batch_size=cfg.batch_size,
#     class_weight=class_weight_map,
#     callbacks=callbacks
# )

# -------- RNN TRAINING SKELETON --------
# df = load_raw_sensor_dataframe()
# values = df[list(cfg.sensor_columns)].values.astype(np.float32)
# y_fall = df['fall_label'].values
#
# X_rnn, y_rnn = sliding_windows(values, y_fall, window_size=RNN_STEPS, stride=cfg.sample_rate_hz)
# X_rnn = np.stack([apply_optional_filter(w, cfg) for w in X_rnn], axis=0)
#
# X_train_rnn, X_val_rnn, y_train_rnn, y_val_rnn = train_test_split(
#     X_rnn, y_rnn, test_size=0.2, random_state=42, stratify=y_rnn
# )
#
# X_train_rnn, X_val_rnn, scaler_rnn = normalize_windows(X_train_rnn, X_val_rnn)
#
# callbacks_rnn = [
#     keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True),
#     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
# ]
#
# hist_rnn = rnn_model.fit(
#     X_train_rnn, y_train_rnn,
#     validation_data=(X_val_rnn, y_val_rnn),
#     epochs=cfg.epochs_rnn,
#     batch_size=cfg.batch_size,
#     callbacks=callbacks_rnn
# )


In [ ]:
def decode_activity(pred_idx: int, class_names: Tuple[str, ...]) -> str:
    return class_names[int(pred_idx)]


def infer_activity_and_fall(
    window_2s: np.ndarray,
    window_20s: Optional[np.ndarray],
    cnn_model: keras.Model,
    rnn_model: keras.Model,
    cfg: PipelineConfig,
    scaler_cnn: Optional[StandardScaler] = None,
    scaler_rnn: Optional[StandardScaler] = None,
    rnn_threshold: float = 0.5
) -> Dict[str, object]:
    w2 = apply_optional_filter(window_2s, cfg)
    if scaler_cnn is not None:
        w2 = scaler_cnn.transform(w2)

    probs = cnn_model.predict(w2[None, ...], verbose=0)[0]
    activity_idx = int(np.argmax(probs))
    peak = detect_fall_peak(window_2s, cfg)

    result = {
        'activity': decode_activity(activity_idx, cfg.activity_classes),
        'activity_probs': probs,
        'fall_peak_triggered': peak,
        'fall_probability': None,
        'fall_confirmed': False,
    }

    if peak and window_20s is not None:
        w20 = apply_optional_filter(window_20s, cfg)
        if scaler_rnn is not None:
            w20 = scaler_rnn.transform(w20)
        p_fall = float(rnn_model.predict(w20[None, ...], verbose=0)[0, 0])
        result['fall_probability'] = p_fall
        result['fall_confirmed'] = p_fall >= rnn_threshold

    return result


def save_models(cnn_model: keras.Model, rnn_model: keras.Model, out_dir: str = 'artifacts'):
    os.makedirs(out_dir, exist_ok=True)
    cnn_path = os.path.join(out_dir, 'cnn_activity_2s.keras')
    rnn_path = os.path.join(out_dir, 'rnn_fall_20s.keras')
    cnn_model.save(cnn_path)
    rnn_model.save(rnn_path)
    print('Saved:', cnn_path)
    print('Saved:', rnn_path)


## 3) Siguientes pasos cuando definas datos

1. Implementar `load_raw_sensor_dataframe()`.
2. Mapear etiquetas de actividad y caida.
3. Ajustar `sample_rate_hz`, umbral de pico y modo de filtro.
4. Entrenar CNN y RNN por separado con validacion por sujeto/sesion.
5. Integrar inferencia encadenada en FastAPI.
